# Runtime YOLO — detecção de cilindros por webcam

Notebook de execução local para o detector YOLO de cilindros.  
Inclui seleção de setup, ajuste de parâmetros, salvamento/sobrescrita de setups e execução em webcam.

**Uso básico:**
1. Execute as células em ordem.
2. Na interface de setups, selecione ou edite os parâmetros.
3. Clique em **Aplicar parâmetros**.
4. Execute a célula **Iniciar runtime da webcam**.
5. Para encerrar, pressione `q` ou `ESC` na janela OpenCV.

In [ ]:
# ============================================================
# Passo 1 — Imports e resolução de caminhos do projeto
# ============================================================

from pathlib import Path
import os
import json
import time
import hashlib
import traceback

import numpy as np
import pandas as pd

try:
    import cv2
except Exception as e:
    raise ImportError(
        "OpenCV não está disponível. Instale com: pip install opencv-python"
    ) from e

try:
    from ultralytics import YOLO
except Exception as e:
    raise ImportError(
        "Ultralytics YOLO não está disponível. Instale com: pip install ultralytics"
    ) from e

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except Exception as e:
    raise ImportError(
        "ipywidgets não está disponível. Instale com: pip install ipywidgets"
    ) from e


def encontrar_raiz_projeto(start=None):
    """Procura a raiz do projeto BLAZE subindo pastas até encontrar src/blaze_paths.py ou vision/."""
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "src" / "blaze_paths.py").exists():
            return p
        if (p / "vision").exists() and (p / "src").exists():
            return p
    return start

PROJECT_ROOT = encontrar_raiz_projeto()

YOLO_RESULTS_DIR = PROJECT_ROOT / "vision" / "results" / "yolo_cylinder_detector"
YOLO_WEIGHTS_DIR = YOLO_RESULTS_DIR / "weights"
YOLO_RUNTIME_RESULTS_DIR = YOLO_RESULTS_DIR / "runtime"
YOLO_RUNTIME_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SETUPS_DIR = PROJECT_ROOT / "vision" / "parameters_setups" / "user" / "cylinders_detect"
SETUPS_DIR.mkdir(parents=True, exist_ok=True)
SETUPS_YOLO_RUNTIME_JSON = SETUPS_DIR / "setups_parametricos_yolo_runtime_user.json"

DEFAULT_MODEL_PATH = YOLO_WEIGHTS_DIR / "yolo_cylinders_best.pt"
DEFAULT_SHA_PATH = YOLO_WEIGHTS_DIR / "yolo_cylinders_best.sha256"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("JSON setups runtime:", SETUPS_YOLO_RUNTIME_JSON)
print("Modelo padrão:", DEFAULT_MODEL_PATH)

In [ ]:
# ============================================================
# Passo 2 — Funções de setup paramétrico
# ============================================================

DEFAULT_RUNTIME_SETUP = {
    "versao": "setups_parametricos_yolo_runtime_v1",
    "descricao": "Setups de runtime YOLO para webcam, com seleção de modelo e ajuste de parâmetros.",
    "setups": {
        "YOLORUN001": {
            "nome": "YOLOv8n runtime padrão — webcam",
            "descricao": "Setup base para inferência YOLO em webcam com o peso treinado do projeto.",
            "params": {
                "model_path": str(DEFAULT_MODEL_PATH),
                "camera_index": 0,
                "conf": 0.35,
                "iou": 0.50,
                "imgsz": 640,
                "max_det": 20,
                "update_every": 1,
                "camera_width": 1280,
                "camera_height": 720,
                "mirror": False,
                "show_fps": True,
                "show_conf": True,
                "line_width": 2,
                "window_name": "BLAZE — YOLO cilindros",
                "device": "",
                "save_runtime_csv": True,
                "save_frames": False,
                "save_video": False,
                "video_fps": 20.0
            }
        }
    }
}


def carregar_json_setups():
    if not SETUPS_YOLO_RUNTIME_JSON.exists() or SETUPS_YOLO_RUNTIME_JSON.stat().st_size == 0:
        salvar_json_setups(DEFAULT_RUNTIME_SETUP)
        return DEFAULT_RUNTIME_SETUP.copy()

    try:
        with open(SETUPS_YOLO_RUNTIME_JSON, "r", encoding="utf-8") as f:
            data = json.load(f)
        if "setups" not in data:
            data["setups"] = {}
        return data
    except Exception:
        print("Erro ao ler JSON de setups. Um arquivo novo será criado.")
        traceback.print_exc()
        salvar_json_setups(DEFAULT_RUNTIME_SETUP)
        return DEFAULT_RUNTIME_SETUP.copy()


def salvar_json_setups(data):
    SETUPS_YOLO_RUNTIME_JSON.parent.mkdir(parents=True, exist_ok=True)
    tmp = SETUPS_YOLO_RUNTIME_JSON.with_suffix(".json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    tmp.replace(SETUPS_YOLO_RUNTIME_JSON)


def lista_opcoes_setup(data):
    setups = data.get("setups", {})
    if not setups:
        return [("Nenhum setup", "")]
    return [
        (f"{sid} — {cfg.get('nome', 'sem nome')}", sid)
        for sid, cfg in setups.items()
    ]


def setup_params(data, sid):
    return data.get("setups", {}).get(sid, {}).get("params", {}).copy()


def calcular_sha256(path, bloco=1024 * 1024):
    path = Path(path)
    if not path.exists():
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(bloco)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def verificar_sha_modelo(model_path):
    model_path = Path(model_path)
    sha_path = model_path.with_suffix(model_path.suffix + ".sha256")

    # Também aceita o nome usado no projeto: yolo_cylinders_best.sha256
    if not sha_path.exists() and model_path.name == "yolo_cylinders_best.pt":
        sha_path = model_path.parent / "yolo_cylinders_best.sha256"

    if not model_path.exists():
        return False, "Modelo não encontrado."

    if not sha_path.exists():
        return None, "Arquivo SHA-256 não encontrado. Validação ignorada."

    try:
        esperado_txt = sha_path.read_text(encoding="utf-8").strip().split()[0]
        atual = calcular_sha256(model_path)
        if atual == esperado_txt:
            return True, "SHA-256 conferido com sucesso."
        return False, "SHA-256 diferente do esperado. Verifique se o peso é o correto."
    except Exception as e:
        return None, f"Não foi possível verificar SHA-256: {e}"


setups_data = carregar_json_setups()
print("Setups carregados:")
for label, sid in lista_opcoes_setup(setups_data):
    print("-", label)

In [ ]:
# ============================================================
# Passo 3 — Interface de seleção, edição e salvamento de setups
# ============================================================

setups_data = carregar_json_setups()

setup_dropdown = widgets.Dropdown(
    options=lista_opcoes_setup(setups_data),
    description="Setup:",
    layout=widgets.Layout(width="760px"),
)

setup_id_txt = widgets.Text(description="ID:", value="YOLORUN001", layout=widgets.Layout(width="350px"))
setup_nome_txt = widgets.Text(description="Nome:", value="YOLOv8n runtime padrão — webcam", layout=widgets.Layout(width="760px"))
setup_desc_txt = widgets.Textarea(description="Descrição:", value="Setup base para inferência YOLO em webcam.", layout=widgets.Layout(width="760px", height="80px"))
overwrite_chk = widgets.Checkbox(description="Sobrescrever setup existente", value=False)

model_path_txt = widgets.Text(description="Modelo:", value=str(DEFAULT_MODEL_PATH), layout=widgets.Layout(width="760px"))
camera_index_int = widgets.IntText(description="Câmera:", value=0, layout=widgets.Layout(width="180px"))
conf_slider = widgets.FloatSlider(description="Conf:", value=0.35, min=0.01, max=0.95, step=0.01, readout_format=".2f", layout=widgets.Layout(width="350px"))
iou_slider = widgets.FloatSlider(description="IoU:", value=0.50, min=0.05, max=0.95, step=0.01, readout_format=".2f", layout=widgets.Layout(width="350px"))
imgsz_int = widgets.IntText(description="Img:", value=640, layout=widgets.Layout(width="180px"))
max_det_int = widgets.IntText(description="Max det:", value=20, layout=widgets.Layout(width="180px"))
update_every_int = widgets.IntSlider(description="Atualizar a cada N frames:", value=1, min=1, max=10, step=1, layout=widgets.Layout(width="420px"))

cam_w_int = widgets.IntText(description="Largura:", value=1280, layout=widgets.Layout(width="220px"))
cam_h_int = widgets.IntText(description="Altura:", value=720, layout=widgets.Layout(width="220px"))
mirror_chk = widgets.Checkbox(description="Espelhar imagem", value=False)
show_fps_chk = widgets.Checkbox(description="Mostrar FPS", value=True)
show_conf_chk = widgets.Checkbox(description="Mostrar confiança", value=True)
line_width_int = widgets.IntSlider(description="Espessura bbox:", value=2, min=1, max=8, step=1, layout=widgets.Layout(width="350px"))
window_name_txt = widgets.Text(description="Janela:", value="BLAZE — YOLO cilindros", layout=widgets.Layout(width="760px"))
device_txt = widgets.Text(description="Device:", value="", placeholder="vazio=auto | 0=GPU 0 | cpu", layout=widgets.Layout(width="350px"))

save_csv_chk = widgets.Checkbox(description="Salvar resumo CSV", value=True)
save_frames_chk = widgets.Checkbox(description="Salvar frames com detecção", value=False)
save_video_chk = widgets.Checkbox(description="Salvar vídeo", value=False)
video_fps_float = widgets.FloatText(description="FPS vídeo:", value=20.0, layout=widgets.Layout(width="220px"))

btn_load = widgets.Button(description="Carregar setup", button_style="info", layout=widgets.Layout(width="180px"))
btn_apply = widgets.Button(description="Aplicar parâmetros", button_style="success", layout=widgets.Layout(width="180px"))
btn_save = widgets.Button(description="Salvar setup", button_style="warning", layout=widgets.Layout(width="180px"))
btn_refresh = widgets.Button(description="Recarregar lista", button_style="", layout=widgets.Layout(width="180px"))

out_setup = widgets.Output()

RUNTIME_PARAMS = {}


def _params_from_widgets():
    return {
        "model_path": model_path_txt.value.strip(),
        "camera_index": int(camera_index_int.value),
        "conf": float(conf_slider.value),
        "iou": float(iou_slider.value),
        "imgsz": int(imgsz_int.value),
        "max_det": int(max_det_int.value),
        "update_every": int(update_every_int.value),
        "camera_width": int(cam_w_int.value),
        "camera_height": int(cam_h_int.value),
        "mirror": bool(mirror_chk.value),
        "show_fps": bool(show_fps_chk.value),
        "show_conf": bool(show_conf_chk.value),
        "line_width": int(line_width_int.value),
        "window_name": window_name_txt.value.strip() or "BLAZE — YOLO cilindros",
        "device": device_txt.value.strip(),
        "save_runtime_csv": bool(save_csv_chk.value),
        "save_frames": bool(save_frames_chk.value),
        "save_video": bool(save_video_chk.value),
        "video_fps": float(video_fps_float.value),
    }


def _widgets_from_setup(sid):
    global setups_data
    cfg = setups_data.get("setups", {}).get(sid)
    if not cfg:
        return

    params = cfg.get("params", {})
    setup_id_txt.value = sid
    setup_nome_txt.value = cfg.get("nome", "")
    setup_desc_txt.value = cfg.get("descricao", "")

    model_path_txt.value = str(params.get("model_path", DEFAULT_MODEL_PATH))
    camera_index_int.value = int(params.get("camera_index", 0))
    conf_slider.value = float(params.get("conf", 0.35))
    iou_slider.value = float(params.get("iou", 0.50))
    imgsz_int.value = int(params.get("imgsz", 640))
    max_det_int.value = int(params.get("max_det", 20))
    update_every_int.value = int(params.get("update_every", 1))
    cam_w_int.value = int(params.get("camera_width", 1280))
    cam_h_int.value = int(params.get("camera_height", 720))
    mirror_chk.value = bool(params.get("mirror", False))
    show_fps_chk.value = bool(params.get("show_fps", True))
    show_conf_chk.value = bool(params.get("show_conf", True))
    line_width_int.value = int(params.get("line_width", 2))
    window_name_txt.value = str(params.get("window_name", "BLAZE — YOLO cilindros"))
    device_txt.value = str(params.get("device", ""))
    save_csv_chk.value = bool(params.get("save_runtime_csv", True))
    save_frames_chk.value = bool(params.get("save_frames", False))
    save_video_chk.value = bool(params.get("save_video", False))
    video_fps_float.value = float(params.get("video_fps", 20.0))


def _refresh_dropdown(selected=None):
    global setups_data
    setups_data = carregar_json_setups()
    setup_dropdown.options = lista_opcoes_setup(setups_data)
    if selected in setups_data.get("setups", {}):
        setup_dropdown.value = selected


def on_load_clicked(_):
    with out_setup:
        clear_output()
        sid = setup_dropdown.value
        if not sid:
            print("Nenhum setup selecionado.")
            return
        _widgets_from_setup(sid)
        print(f"Setup carregado: {sid}")


def on_apply_clicked(_):
    global RUNTIME_PARAMS
    with out_setup:
        clear_output()
        RUNTIME_PARAMS = _params_from_widgets()
        ok_sha, msg_sha = verificar_sha_modelo(RUNTIME_PARAMS["model_path"])
        print("Parâmetros aplicados.")
        print("Setup atual:", setup_id_txt.value, "|", setup_nome_txt.value)
        print("Modelo:", RUNTIME_PARAMS["model_path"])
        print("Conf:", RUNTIME_PARAMS["conf"], "| IoU:", RUNTIME_PARAMS["iou"], "| Img:", RUNTIME_PARAMS["imgsz"])
        print("Câmera:", RUNTIME_PARAMS["camera_index"], "| Resolução:", RUNTIME_PARAMS["camera_width"], "x", RUNTIME_PARAMS["camera_height"])
        print("Verificação do modelo:", msg_sha)


def on_save_clicked(_):
    global setups_data
    with out_setup:
        clear_output()
        sid = setup_id_txt.value.strip()
        if not sid:
            print("Defina um ID para o setup.")
            return

        setups_data = carregar_json_setups()
        existe = sid in setups_data.get("setups", {})
        if existe and not overwrite_chk.value:
            print(f"O setup {sid} já existe. Marque 'Sobrescrever setup existente' para substituir.")
            return

        setups_data.setdefault("setups", {})[sid] = {
            "nome": setup_nome_txt.value.strip(),
            "descricao": setup_desc_txt.value.strip(),
            "params": _params_from_widgets(),
        }
        salvar_json_setups(setups_data)
        _refresh_dropdown(selected=sid)
        print(f"Setup salvo: {sid}")
        print("Arquivo:", SETUPS_YOLO_RUNTIME_JSON)


def on_refresh_clicked(_):
    with out_setup:
        clear_output()
        _refresh_dropdown()
        print("Lista recarregada.")

btn_load.on_click(on_load_clicked)
btn_apply.on_click(on_apply_clicked)
btn_save.on_click(on_save_clicked)
btn_refresh.on_click(on_refresh_clicked)

# Carrega o primeiro setup automaticamente.
if setup_dropdown.value:
    _widgets_from_setup(setup_dropdown.value)
    RUNTIME_PARAMS = _params_from_widgets()

interface = widgets.VBox([
    widgets.HTML("<h3>Selecionar setup existente</h3>"),
    setup_dropdown,
    widgets.HBox([btn_load, btn_apply, btn_save, btn_refresh]),
    widgets.HTML("<hr><h3>Identificação do setup</h3>"),
    setup_id_txt,
    setup_nome_txt,
    setup_desc_txt,
    overwrite_chk,
    widgets.HTML("<hr><h3>Parâmetros principais</h3>"),
    model_path_txt,
    widgets.HBox([camera_index_int, imgsz_int, max_det_int]),
    widgets.HBox([conf_slider, iou_slider]),
    update_every_int,
    widgets.HTML("<hr><h3>Webcam e visualização</h3>"),
    widgets.HBox([cam_w_int, cam_h_int]),
    widgets.HBox([mirror_chk, show_fps_chk, show_conf_chk]),
    line_width_int,
    window_name_txt,
    device_txt,
    widgets.HTML("<hr><h3>Registro opcional</h3>"),
    widgets.HBox([save_csv_chk, save_frames_chk, save_video_chk, video_fps_float]),
    out_setup,
])

display(interface)

In [ ]:
# ============================================================
# Passo 4 — Funções do runtime YOLO
# ============================================================


def carregar_modelo_yolo(params):
    model_path = Path(params.get("model_path", DEFAULT_MODEL_PATH)).expanduser()
    if not model_path.exists():
        raise FileNotFoundError(f"Modelo YOLO não encontrado: {model_path}")

    ok_sha, msg_sha = verificar_sha_modelo(model_path)
    print("Carregando modelo:", model_path)
    print("Verificação SHA:", msg_sha)

    model = YOLO(str(model_path))
    return model


def desenhar_resultado(frame, result, params, fps=None):
    """Usa o plot do Ultralytics e acrescenta FPS se habilitado."""
    line_width = int(params.get("line_width", 2))
    show_conf = bool(params.get("show_conf", True))

    try:
        annotated = result.plot(
            line_width=line_width,
            conf=show_conf,
            labels=True,
        )
    except TypeError:
        annotated = result.plot()

    if params.get("show_fps", True) and fps is not None:
        txt = f"FPS: {fps:.1f}"
        cv2.putText(
            annotated,
            txt,
            (12, 32),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

    return annotated


def salvar_resumo_runtime(registros, params):
    if not registros:
        return None

    df = pd.DataFrame(registros)
    ts = time.strftime("%Y%m%d_%H%M%S")
    out_csv = YOLO_RUNTIME_RESULTS_DIR / f"runtime_yolo_webcam_{ts}.csv"
    df.to_csv(out_csv, index=False)

    setup_csv = YOLO_RUNTIME_RESULTS_DIR / f"runtime_yolo_webcam_setup_{ts}.json"
    with open(setup_csv, "w", encoding="utf-8") as f:
        json.dump(params, f, indent=2, ensure_ascii=False)

    return out_csv


def abrir_camera(camera_index, width=None, height=None):
    cap = cv2.VideoCapture(int(camera_index))
    if width:
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, int(width))
    if height:
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, int(height))
    return cap


def iniciar_runtime_yolo(params=None):
    """
    Inicia runtime YOLO em janela OpenCV.
    Pressione q ou ESC para sair.
    """
    params = dict(params or RUNTIME_PARAMS or _params_from_widgets())

    model = carregar_modelo_yolo(params)

    camera_index = int(params.get("camera_index", 0))
    cap = abrir_camera(
        camera_index,
        params.get("camera_width", None),
        params.get("camera_height", None),
    )

    if not cap.isOpened():
        raise RuntimeError(f"Não foi possível abrir a câmera índice {camera_index}.")

    window_name = params.get("window_name", "BLAZE — YOLO cilindros")
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    writer = None
    if params.get("save_video", False):
        ts = time.strftime("%Y%m%d_%H%M%S")
        video_path = YOLO_RUNTIME_RESULTS_DIR / f"runtime_yolo_webcam_{ts}.mp4"
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        fps_video = float(params.get("video_fps", 20.0))
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or int(params.get("camera_width", 1280))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or int(params.get("camera_height", 720))
        writer = cv2.VideoWriter(str(video_path), fourcc, fps_video, (w, h))
        print("Gravando vídeo em:", video_path)

    registros = []
    last_result = None
    last_annotated = None
    frame_idx = 0
    t_start = time.time()
    t_prev = time.time()
    fps_smooth = None

    print("Runtime iniciado.")
    print("Pressione q ou ESC na janela OpenCV para sair.")

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                print("Frame não capturado. Encerrando.")
                break

            if params.get("mirror", False):
                frame = cv2.flip(frame, 1)

            t_now = time.time()
            dt = max(t_now - t_prev, 1e-6)
            fps_inst = 1.0 / dt
            t_prev = t_now
            fps_smooth = fps_inst if fps_smooth is None else (0.9 * fps_smooth + 0.1 * fps_inst)

            update_every = max(1, int(params.get("update_every", 1)))
            rodar_inferencia = (frame_idx % update_every == 0) or (last_result is None)

            infer_s = np.nan
            n_det = np.nan

            if rodar_inferencia:
                infer_t0 = time.time()
                predict_kwargs = {
                    "source": frame,
                    "conf": float(params.get("conf", 0.35)),
                    "iou": float(params.get("iou", 0.50)),
                    "imgsz": int(params.get("imgsz", 640)),
                    "max_det": int(params.get("max_det", 20)),
                    "verbose": False,
                }
                device = str(params.get("device", "")).strip()
                if device:
                    predict_kwargs["device"] = device

                results = model.predict(**predict_kwargs)
                infer_s = time.time() - infer_t0
                last_result = results[0]

                try:
                    n_det = len(last_result.boxes) if last_result.boxes is not None else 0
                except Exception:
                    n_det = np.nan

                last_annotated = desenhar_resultado(frame, last_result, params, fps=fps_smooth)
            else:
                if last_annotated is not None:
                    last_annotated = last_annotated.copy()
                    if params.get("show_fps", True):
                        cv2.putText(
                            last_annotated,
                            f"FPS: {fps_smooth:.1f}",
                            (12, 32),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9,
                            (255, 255, 255),
                            2,
                            cv2.LINE_AA,
                        )
                else:
                    last_annotated = frame

            cv2.imshow(window_name, last_annotated)

            if writer is not None:
                writer.write(last_annotated)

            if params.get("save_frames", False) and rodar_inferencia:
                frames_dir = YOLO_RUNTIME_RESULTS_DIR / "frames"
                frames_dir.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(frames_dir / f"frame_{frame_idx:06d}.jpg"), last_annotated)

            registros.append({
                "frame": frame_idx,
                "tempo_desde_inicio_s": time.time() - t_start,
                "fps_smooth": fps_smooth,
                "inferencia_rodada": rodar_inferencia,
                "tempo_inferencia_s": infer_s,
                "n_deteccoes": n_det,
                "conf": params.get("conf", np.nan),
                "iou": params.get("iou", np.nan),
                "imgsz": params.get("imgsz", np.nan),
            })

            key = cv2.waitKey(1) & 0xFF
            if key in [27, ord("q")]:
                break

            frame_idx += 1

    finally:
        cap.release()
        if writer is not None:
            writer.release()
        cv2.destroyAllWindows()

        if params.get("save_runtime_csv", True):
            out_csv = salvar_resumo_runtime(registros, params)
            if out_csv:
                print("Resumo runtime salvo em:", out_csv)

        print("Runtime encerrado.")
        print("Frames processados:", frame_idx)
        if fps_smooth is not None:
            print(f"FPS final aproximado: {fps_smooth:.2f}")

In [ ]:
# ============================================================
# Passo 5 — Iniciar runtime da webcam
# ============================================================

# Antes de executar esta célula, configure os parâmetros no Passo 3
# e clique em "Aplicar parâmetros".

iniciar_runtime_yolo(RUNTIME_PARAMS)